# 02 — Làm sạch dữ liệu và phân tích mô tả

**Đầu vào và đầu ra**

Dữ liệu thô của Giai đoạn 1 được xử lý thành corpus trung gian trong `01_data/interim/`, là dữ liệu dùng để gán nhãn và huấn luyện mô
hình. File thô không bị sửa.

**Shopee và Lazada**

Hai app được xử lý giống nhau nhưng lưu riêng, vì Lazada chỉ đóng vai trò External Test.

### Các bước xử lý (theo thứ tự)

**1. Kiểm tra trùng `review_id`**

Kiểm tra lại dù bước gộp ở Giai đoạn 1 đã khử trùng.

**2. Cắt cửa sổ nghiên cứu**

Cắt đúng cửa sổ từ 06/10/2025 00:00 đến 06/10/2026 23:59 (giờ VN). Lô cuối mỗi lần crawl thường lấy quá mốc ngày một chút nên phải cắt
lại ở bước này.

**3. Ẩn danh hóa lại**

Áp lại luật ẩn danh hóa mới nhất cho `text` và `reply_content`.

**4. Chuẩn hóa tối thiểu**

Chuẩn Unicode NFC, bỏ ký tự điều khiển, gộp khoảng trắng thừa. Văn bản cố ý không được chuyển chữ thường, không bỏ emoji, không sửa
chính tả, vì ViSoBERT được huấn luyện trên văn bản mạng xã hội và người gán nhãn cũng cần đọc đúng cách người dùng viết (đề cương
§7.1).

**5. Loại review không có chữ**

Bỏ những review rỗng hoặc không có chữ cái nào (chỉ emoji, ký hiệu hoặc số); các dòng bị bỏ được lưu riêng kèm lý do.

**6. Đánh dấu thay vì xóa**

Review có nội dung trùng y hệt review khác (`near_duplicate`) và review có thể không phải tiếng Việt (`non_vi`, `lang_und`) chỉ được
đánh dấu, không bị xóa.

**Hai trường văn bản**

Mỗi bản ghi giữ cả `text` (văn bản gốc đã ẩn danh hóa, để truy vết) lẫn `text_clean` (bản đã chuẩn hóa).

**Vị trí code**

Phần xử lý nằm trong `src/preprocessing/clean.py` và `src/preprocessing/eda.py`.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath(os.path.join('..', '02_src')))
from src.utils.kaggle_paths import input_dir, working_dir
from src.utils.seeds import set_seed
from src.preprocessing.clean import (clean_app, load_jsonl, pii_qa, write_manifest,
                                     WINDOW_START, WINDOW_END)
from src.preprocessing import eda
set_seed(42)
TAG = 'T0_2026-10-06'
RAW, INTERIM, MANIFESTS = input_dir('raw'), input_dir('interim'), input_dir('manifests')
EDA_DIR = working_dir('eda')
FIG_DIR = working_dir(os.path.join('eda', 'figures'))
SOURCES = {
    'shopee': os.path.join(RAW, 'shopee', f'shopee_reviews_stratified_{TAG}.jsonl'),
    'lazada': os.path.join(RAW, 'lazada', f'lazada_reviews_{TAG}_r2.jsonl'),
}
print('Cửa sổ nghiên cứu:', WINDOW_START.isoformat(), '→', WINDOW_END.isoformat())

Cửa sổ nghiên cứu: 2025-10-06T00:00:00+07:00 → 2026-10-06T23:59:59+07:00


## Làm sạch và cắt cửa sổ thời gian

In [2]:
runs, records = {}, {}
for app, raw_path in SOURCES.items():
    out = os.path.join(INTERIM, f'{app}_clean_{TAG}.jsonl')
    dropped = os.path.join(INTERIM, f'{app}_dropped_{TAG}.jsonl')
    runs[app] = clean_app(raw_path, app, out, dropped)
    records[app] = load_jsonl(out)

labels = [('Raw', 'raw'), ('Trùng review_id', 'duplicate_review_id'),
          ('Ngoài cửa sổ (trước 2025-10-06)', 'outside_window_before'),
          ('Ngoài cửa sổ (sau 2026-10-06 23:59)', 'outside_window_after'),
          ('Sau cắt cửa sổ', 'after_window'), ('Loại: text rỗng', 'drop_empty'),
          ('Loại: không có chữ cái', 'drop_no_letters'), ('Sau lọc (interim)', 'after_filter'),
          ('Thay PII bổ sung ở interim', 'pii_replacements_at_interim')]
print(f"{'Bước':40s}{'Shopee':>10s}{'Lazada':>10s}")
for label, k in labels:
    print(f"{label:40s}{runs['shopee']['steps'][k]:>10,}{runs['lazada']['steps'][k]:>10,}")

Bước                                        Shopee    Lazada
Raw                                         57,200     3,600
Trùng review_id                                  0         0
Ngoài cửa sổ (trước 2025-10-06)                831        94
Ngoài cửa sổ (sau 2026-10-06 23:59)              0         0
Sau cắt cửa sổ                              56,369     3,506
Loại: text rỗng                                  0         0
Loại: không có chữ cái                         319        12
Sau lọc (interim)                           56,050     3,494
Thay PII bổ sung ở interim                       5     1,105


## Manifest

**Nội dung ghi lại**

Số lượng qua từng bước, khoảng ngày và checksum của file nguồn lẫn file kết quả.

**Kiểm tra cửa sổ**

Khoảng ngày của corpus trung gian phải nằm gọn trong cửa sổ nghiên cứu.

In [3]:
from datetime import datetime
manifest_path = os.path.join(MANIFESTS, f'gd2_interim_manifest_{TAG}.json')
manifest = write_manifest(list(runs.values()), manifest_path)
for app, r in runs.items():
    lo, hi = datetime.fromisoformat(r['min_review_date']), datetime.fromisoformat(r['max_review_date'])
    assert WINDOW_START <= lo and hi <= WINDOW_END, f'{app}: ngoài cửa sổ'
    print(f"{app:7s} min={lo.astimezone(WINDOW_START.tzinfo)} max={hi.astimezone(WINDOW_START.tzinfo)} "
          f"| raw sha256={r['raw_sha256'][:12]}… | interim sha256={r['out_sha256'][:12]}…")
print('Manifest:', manifest_path)

shopee  min=2025-10-06 00:03:56+07:00 max=2026-10-06 07:18:03+07:00 | raw sha256=d8115c1fd7d9… | interim sha256=42d1e4811fc5…
lazada  min=2025-10-06 06:09:42+07:00 max=2026-10-06 05:45:41+07:00 | raw sha256=c7c882890de9… | interim sha256=1cd44b596fff…
Manifest: C:\Workspace\kltn-vn-app-review-requirements\01_data\manifests\gd2_interim_manifest_T0_2026-10-06.json


## Kiểm tra lại phần ẩn danh hóa

**Lý do**

Việc thay tên dựa hoàn toàn trên luật, nên cần kiểm tra xem còn sót tên, số điện thoại hay email không.

**Mẫu đọc trực tiếp**

Với mỗi app, lấy ngẫu nhiên (seed 42) 50 phản hồi của sàn và 30 review để đọc trực tiếp.

**Regex dò trên toàn bộ dữ liệu**

Chuỗi 2–3 từ viết hoa đầu từ sau "bạn/anh/chị/em", dãy từ 9 chữ số trở lên, và chuỗi có ký tự `@`.

In [4]:
qa = {}
for app in SOURCES:
    qa[app] = pii_qa(records[app], n_reply=50, n_text=30, seed=42)
    q = qa[app]
    print(f'===== {app}: mẫu reply={q["n_sample_reply"]}, mẫu text={q["n_sample_text"]}, '
          f'nghi sót trong mẫu={q["n_suspects_in_sample"]}')
    print('Nghi sót trên toàn corpus:', q['suspects_full_corpus'] or 'không có')
    for kind, ex in q['suspect_examples'].items():
        print(f'  {kind}:', [frag for _, frag in ex])

===== shopee: mẫu reply=50, mẫu text=30, nghi sót trong mẫu=0
Nghi sót trên toàn corpus: {'reply_content:long_digits': 3, 'text:long_digits': 8, 'text:title_after_pronoun': 10}
  reply_content:long_digits: ['1441554913', '1441554913', '1441554913']
  text:long_digits: ['01-16-27-34-48-54', '066991025085', '61 7444444441', '100000000', '1000000000000000', '1000000000.000000', '1000000000', '10000000000000000']
  text:title_after_pronoun: ['Anh Không Làm Được', 'Em Cuối Chiều Em', 'Anh Vực Sâu Tối', 'Em Chẳng Màng Chỉ', 'Anh Ngỡ Ngàng Em', 'Anh Đã Thấy Đôi', 'Em Mỉm Cười Bên', 'Anh Giấu Đi Lệ']
===== lazada: mẫu reply=50, mẫu text=30, nghi sót trong mẫu=0
Nghi sót trên toàn corpus: không có


In [5]:
# Mẫu soi trực tiếp (cắt 160 ký tự mỗi dòng)
for app in SOURCES:
    print(f'===== {app} — reply_content')
    for rid, s in qa[app]['sample_reply']:
        print(f'  {rid[:8]} | {s[:160]}')
    print(f'===== {app} — text')
    for rid, s in qa[app]['sample_text']:
        print(f'  {rid[:8]} | {s[:160]}')

===== shopee — reply_content
  4123db46 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  09f05b73 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  b8c0ae57 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  f0dfede2 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  e0e8b8af | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  3756aba0 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụ

**Kết quả kiểm tra**

Phần kiểm tra này quan trọng hơn dự kiến: luật ẩn danh hóa phải sửa hai lần.

**Lần sửa 1 — tên sót do regex phát hiện**

Bộ regex dò ra một số tên còn sót trong phản hồi của sàn. Ở Shopee là những tên hiển thị sau "Chào bạn" bắt đầu bằng chữ như "Anh",
"Nha", "Vui" (ví dụ "Anh MvP", "Nha Thanh") mà luật cũ tưởng là đại từ. Ở Lazada là những phản hồi mở đầu thẳng bằng tên kiểu "<tên>,
cảm ơn bạn…", và có cả một phản hồi bằng tiếng Na Uy ("Hei <tên>!").

**Lần sửa 2 — tên sót khi đọc trực tiếp**

Sau lần sửa 1, đọc lại 50 phản hồi mẫu của Lazada vẫn thấy 3 tên còn sót mà regex không bắt được, do câu sau tên không phải "cảm ơn" mà
là ", chúng tôi…", ", mình…" hoặc tiếng Anh. Luật được mở rộng cho các trường hợp này.

**Phạm vi áp luật mới**

Luật mới được thêm vào hàm `scrub_pii` (`src/collect/google_play.py`) và chỉ áp cho phản hồi của sàn.

**Kiểm tra thay thừa**

Sau mỗi lần sửa, toàn bộ những đoạn bị thay đều được xem lại để chắc là chỉ thay đúng tên. Trường hợp thay thừa duy nhất là "Kakak",
một cách xưng hô tiếng Indonesia, không gây hại gì.

**Báo nhầm còn lại ở Shopee**

Những gì regex còn báo ở Shopee đều là báo nhầm: số tiền, mã vận đơn `SPXVN…`, mã sự kiện trong đường link của Shopee, và lời một bài
hát viết hoa từng chữ.

**Hệ quả với dữ liệu thô**

File thô không được sửa nên vẫn còn các tên bị sót; từ đây trở đi chỉ dùng dữ liệu trung gian.

## Phân tích mô tả

**Chưa dùng nhãn**

Ở giai đoạn này chưa có nhãn, nên dữ liệu chỉ được mô tả; tỷ lệ BUG/FEATURE/UX hay tỷ lệ actionable không được ước lượng — kể cả bằng
từ khóa — để không ảnh hưởng tới việc gán nhãn sau này.

**Múi giờ**

Mọi mốc thời gian được quy về giờ Việt Nam.

In [6]:
dfs = {app: eda.to_frame(records[app], app) for app in SOURCES}
summaries = {app: eda.summarize(dfs[app], runs[app]['steps']['drop_no_letters'], runs[app]['steps']['after_window'])
             for app in SOURCES}
for app, s in summaries.items():
    print(f'===== {app}')
    print(json.dumps(s, ensure_ascii=False, indent=1, default=str))

===== shopee
{
 "n_reviews": 56050,
 "date_min": "2025-10-06 00:03:56+07:00",
 "date_max": "2026-10-06 07:18:03+07:00",
 "rating_pct": {
  "1": 39.9,
  "2": 3.4,
  "3": 3.2,
  "4": 5.0,
  "5": 48.5
 },
 "words_q25_q50_q75_q90": [
  2.0,
  6.0,
  16.0,
  32.0
 ],
 "chars_median": 27.0,
 "short_le2_words_pct": 28.6,
 "app_version_missing_pct": 26.3,
 "top_versions": {
  "3.65.22": 2729,
  "3.61.28": 2416,
  "3.81.32": 2254,
  "3.77.25": 2107,
  "3.66.26": 1828
 },
 "reply_pct": 60.3,
 "lang_pct": {
  "vi": 90.9,
  "und": 8.8,
  "non_vi": 0.3
 },
 "near_dup_pct": 30.1,
 "no_letters_dropped_pct": 0.57
}
===== lazada
{
 "n_reviews": 3494,
 "date_min": "2025-10-06 06:09:42+07:00",
 "date_max": "2026-10-06 05:45:41+07:00",
 "rating_pct": {
  "1": 60.7,
  "2": 3.5,
  "3": 3.8,
  "4": 3.7,
  "5": 28.3
 },
 "words_q25_q50_q75_q90": [
  4.0,
  11.0,
  25.0,
  47.0
 ],
 "chars_median": 47.0,
 "short_le2_words_pct": 19.3,
 "app_version_missing_pct": 34.0,
 "top_versions": {
  "7.88.0": 221,
  "7.89

In [7]:
figures = eda.all_figures(dfs, FIG_DIR)
for f in figures:
    print(f)

C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_rating_distribution.png
C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_length_words.png
C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_monthly_volume.png
C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_daily_tail.png
C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_app_version_top.png


## Nhận xét

**Cơ sở**

Phần nhận xét dưới đây được viết sau khi đọc các số liệu và hình ở trên.

**Báo cáo đầy đủ**

Bảng đối soát, bảng tóm tắt, kiểm tra ẩn danh hóa, hạn chế và danh sách hình được xuất ra `05_experiments/eda/eda_report.md`.

In [8]:
NOTES = {
    'shopee': (
        'Sau khi làm sạch còn 56.050 review trong 12 tháng. Điều tôi thấy rõ nhất là đánh giá rất phân cực: 1★ và 5★ chiếm gần 90%, '
        'các mức 2–4★ chỉ khoảng 11–12%. Mỗi tháng có khoảng 3.500–6.300 review, nhiều nhất vào 01/2026. Phần lớn review rất ngắn: '
        'trung vị chỉ 6 từ, gần 29% có không quá 2 từ, và khoảng 30% review trùng y hệt nội dung với review khác, chủ yếu là những câu '
        'như "ok", "tốt", "rất tốt". Khoảng 60% review có phản hồi của sàn; 26,3% không có thông tin phiên bản app.'),
    'lazada': (
        'Lazada có 3.494 review, ít hơn Shopee rất nhiều nhưng vẫn đủ cho External Test. Review ở đây tiêu cực hơn hẳn: 1★ khoảng 61%, '
        '5★ khoảng 28%. Mỗi tháng có khoảng 190–520 review, nhiều nhất vào 12/2025. Người dùng Lazada viết dài hơn (trung vị 11 từ, '
        'khoảng 19% có không quá 2 từ) và ít trùng nội dung hơn (khoảng 15%). Sàn trả lời mọi review và thường gọi tên người dùng, '
        'đó cũng là lý do Lazada có nhiều chỗ phải ẩn danh hóa. 34,0% không có thông tin phiên bản app.'),
    'qa_action': (
        'Tôi sửa luật ẩn danh hóa cho phản hồi của sàn hai lần. Lần 1 (từ regex dò): bắt thêm tên sau "Chào bạn" bắt đầu bằng chữ như '
        '"Anh", "Nha", "Vui", và phản hồi mở đầu thẳng bằng tên trước ", cảm ơn". Lần 2 (từ việc tự đọc mẫu, 3/50 phản hồi Lazada còn '
        'sót): bắt thêm tên đứng đầu trước ", chúng tôi", ", mình" hoặc câu tiếng Anh. Sau hai lần sửa, mẫu đọc lại và regex dò không '
        'còn thấy tên sót ở Lazada; các trường hợp regex còn báo ở Shopee tôi đã kiểm tra từng cái và đều là báo nhầm. File thô giữ '
        'nguyên và không đưa lên Git; từ đây tôi chỉ dùng dữ liệu trung gian.'),
}
report = eda.write_report(os.path.join(EDA_DIR, 'eda_report.md'), summaries,
                          {a: runs[a]['steps'] for a in SOURCES}, qa, figures, NOTES)
print('Báo cáo:', report)

Báo cáo: C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\eda_report.md


## Tổng kết Giai đoạn 2

In [9]:
print('Đối soát số lượng (raw → sau cắt cửa sổ → sau lọc):')
for app, r in runs.items():
    s = r['steps']
    print(f"  {app:7s} {s['raw']:,} → {s['after_window']:,} (ngoài cửa sổ: {s['outside_window_before'] + s['outside_window_after']:,}) "
          f"→ {s['after_filter']:,} (rỗng: {s['drop_empty']}, không có chữ cái: {s['drop_no_letters']})")
print('Khoảng ngày interim nằm trong cửa sổ: có (đã kiểm ở mục 4)')
print('QA ẩn danh hóa:')
for app, q in qa.items():
    print(f"  {app:7s} mẫu soi: {q['n_sample_reply']} reply + {q['n_sample_text']} text | nghi sót trong mẫu: "
          f"{q['n_suspects_in_sample']} | nghi sót toàn corpus (regex phụ): {sum(q['suspects_full_corpus'].values())}")
print('Hình đã lưu:')
for f in figures:
    print('  ', os.path.relpath(f, os.path.join('..')))

Đối soát số lượng (raw → sau cắt cửa sổ → sau lọc):
  shopee  57,200 → 56,369 (ngoài cửa sổ: 831) → 56,050 (rỗng: 0, không có chữ cái: 319)
  lazada  3,600 → 3,506 (ngoài cửa sổ: 94) → 3,494 (rỗng: 0, không có chữ cái: 12)
Khoảng ngày interim nằm trong cửa sổ: có (đã kiểm ở mục 4)
QA ẩn danh hóa:
  shopee  mẫu soi: 50 reply + 30 text | nghi sót trong mẫu: 0 | nghi sót toàn corpus (regex phụ): 21
  lazada  mẫu soi: 50 reply + 30 text | nghi sót trong mẫu: 0 | nghi sót toàn corpus (regex phụ): 0
Hình đã lưu:
   05_experiments\eda\figures\eda_rating_distribution.png
   05_experiments\eda\figures\eda_length_words.png
   05_experiments\eda\figures\eda_monthly_volume.png
   05_experiments\eda\figures\eda_daily_tail.png
   05_experiments\eda\figures\eda_app_version_top.png


## Hạn chế

**Ngày 06/10/2026 chưa đủ dữ liệu**

Ngày này chỉ có dữ liệu tới khoảng 7 giờ sáng (giờ VN), vì Google Play hiển thị review trễ khoảng một ngày. Dữ liệu ngày này không được
bù; nó chỉ ảnh hưởng nhẹ tới cửa sổ 30 ngày gần nhất khi phân tích theo thời gian.

**Xác định ngôn ngữ bằng luật**

Ngôn ngữ được xác định bằng luật (dấu tiếng Việt, chữ ngoài hệ Latin, một từ điển nhỏ các từ tiếng Việt không dấu). Những review rất
ngắn không dấu như "ok", "good" được để là `und`. Review không phải tiếng Việt mới chỉ được đánh dấu, chưa bị loại.

**Phạm vi của `near_duplicate`**

Chỉ bắt các review trùng y hệt sau chuẩn hóa; những biến thể khác dấu câu hay chữ hoa không được tính.

**Ẩn danh hóa có thể còn sót**

Dù đã kiểm tra kỹ, việc ẩn danh hóa vẫn có thể sót ở những vị trí bất thường mà mẫu kiểm tra chưa gặp.